# Transit demand forecasting — walkthrough

This notebook explores the data and the results produced by `scripts/run_experiments.py`.
Training happens in that script (it takes a while and saves everything to `outputs/`), so run it first:

```bash
python scripts/run_experiments.py
```

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np, pandas as pd, matplotlib.pyplot as plt
import config
from utils.dataset import prepare_data
from utils.metrics import compute_metrics
from models.baselines import seasonal_naive, hour_of_week_mean

## 1. Data
Hourly boardings per stop, split chronologically: Oct–Jan train, February validation, March test.

In [ ]:
df = pd.read_parquet(config.DATASET_PATH)
data = prepare_data(df)
print(data.stops)

In [ ]:
route_total = pd.Series(data.y_raw.sum(axis=0), index=data.times)
ax = route_total.resample("D").sum().plot(figsize=(12, 3), title="Daily boardings, forecast stops")
for name, start in [("val", config.VAL_START), ("test", config.TEST_START)]:
    ax.axvline(pd.Timestamp(start), color="k", ls="--"); ax.text(pd.Timestamp(start), ax.get_ylim()[1], f" {name}", va="top")

In [ ]:
prof = route_total.groupby([route_total.index.dayofweek, route_total.index.hour]).mean().unstack(0)
prof.columns = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
prof.plot(figsize=(10, 3.5), title="Average hourly boardings by day of week"); plt.xlabel("hour");

## 2. Baselines
Any model has to beat these simple forecasts on the test month.

In [ ]:
o = data.splits["test"]
y = data.window(o, "y_raw")
pd.DataFrame({
    "same hour yesterday": compute_metrics(y, seasonal_naive(data, o, 24)),
    "same hour last week": compute_metrics(y, seasonal_naive(data, o, 168)),
    "hour-of-week mean":   compute_metrics(y, hour_of_week_mean(data, o)),
}).T.round(2)

## 3. Results from `run_experiments.py`

In [ ]:
metrics = pd.read_csv(config.RESULTS_DIR / "metrics.csv")
cols = ["model", "MAE", "RMSE", "WMAPE"] + [c for c in metrics.columns if c.startswith("MAPE(") and not c.endswith("_mean")]
metrics[cols]

In [ ]:
summary = json.loads((config.RESULTS_DIR / "summary.json").read_text())
print("Ensemble weights:", {k: round(v, 2) for k, v in summary["weights"].items() if v > 0})
print("Residual corrector:", summary["residual"])
if summary.get("ablation"):
    display(pd.DataFrame(summary["ablation"])[["variant", "test_WMAPE"]])

In [ ]:
from IPython.display import Image, display
for name in ["model_comparison", "error_by_hour_Ensemble", "error_by_stop_Ensemble", "forecast_day_Ensemble"]:
    display(Image(config.PLOT_DIR / f"{name}.png"))

## 4. One day through the whole system
Forecast → segment loads → frequency plan, exactly as the API does it.

In [ ]:
from backend.inference import get_engine
from optimize.frequency import summary_table
engine = get_engine()
sim = engine.run_simulation(day_index=0, bus_capacity=config.BUS_CAPACITY, fixed_headway=config.FIXED_HEADWAY_MIN)
print(sim["day"], "— day WMAPE", sim["forecast"]["day_wmape"], "%")
plt.figure(figsize=(10, 3)); plt.plot(sim["forecast"]["actual"], label="actual"); plt.plot(sim["forecast"]["predicted"], "--", label="forecast"); plt.legend(); plt.xlabel("hour");

In [ ]:
heat = pd.DataFrame(sim["heatmap"]["data"], index=sim["heatmap"]["segments"], columns=sim["heatmap"]["hours"])
plt.figure(figsize=(12, 6)); plt.imshow(heat, aspect="auto", cmap="magma"); plt.colorbar(label="passengers / hour")
plt.yticks(range(len(heat)), heat.index, fontsize=7); plt.xticks(range(24), heat.columns); plt.title("Segment load");
opt = sim["optimization"]
print(f"Cost change {-opt['savings']['cost_pct']:+.1f}%, wait change {opt['savings']['wait_change_min']:+.2f} min, "
      f"overcrowded hours {opt['fixed']['overcrowded_hours']} → {opt['optimized']['overcrowded_hours']}")